# 准备 TorchTitan-NPU 运行环境与训练资产

本节先获取训练代码，再调用安装脚本创建独立的 TorchTitan-NPU 环境。脚本会固定源码版本、应用适配补丁、创建 uv 虚拟环境、安装依赖，并在结束前检查环境是否可用。

第一次执行需要下载和构建多个依赖。如果工作区中没有可复用的训练资产，Cell 还会生成 2000/20 条 Wordle parquet，并下载 Qwen3-1.7B Wordle SFT 权重；再次执行时会跳过已经完成的源码、环境和资产。

同级目录中已有 `cann-recipes-train` 时，Cell 直接使用当前分支和本地修改；目录不存在时，才会克隆官方 `master` 分支。


## 运行下面的 Cell

Cell 会获取或复用同级的训练代码，执行 `setup_backend.sh`，并补齐缺失的模型或数据。正常结束时会打印训练资产目录；该目录中应包含完整模型以及训练、验证两个 parquet 文件。

如果执行中止，请先查看最后一条错误。常见原因包括 Python、GCC 或 CANN 版本不匹配，网络或磁盘空间不足，依赖安装失败，以及双 NPU 不可见。修复对应问题后可以直接重新运行，已完成的内容会被复用。


In [ ]:
%%bash
set -euo pipefail

COURSE_ROOT=$(git rev-parse --show-toplevel)
WORKSPACE_ROOT=$(dirname "${COURSE_ROOT}")
RECIPE_ROOT="${WORKSPACE_ROOT}/cann-recipes-train"
RECIPE_REPO_URL="${RECIPE_REPO_URL:-https://gitcode.com/cann/cann-recipes-train.git}"
RECIPE_BRANCH="${RECIPE_BRANCH:-master}"

if [[ -d "${RECIPE_ROOT}/.git" ]]; then
    CURRENT_RECIPE_BRANCH=$(git -C "${RECIPE_ROOT}" branch --show-current)
    CURRENT_RECIPE_COMMIT=$(git -C "${RECIPE_ROOT}" rev-parse --short HEAD)
    printf '复用已有训练代码目录: %s (%s@%s)\n' \
        "${RECIPE_ROOT}" "${CURRENT_RECIPE_BRANCH:-detached}" "${CURRENT_RECIPE_COMMIT}"
else
    git clone --branch "${RECIPE_BRANCH}" "${RECIPE_REPO_URL}" "${RECIPE_ROOT}"
fi

TRAIN_DIR="${RECIPE_ROOT}/llm_rl/qwen3_wordle"
BACKEND_DIR="${TRAIN_DIR}/torchtitan_backend"
test -f "${TRAIN_DIR}/torchtitan_backend/setup_backend.sh" || {
    printf '当前训练代码缺少 TorchTitan-NPU 后端，请更新代码后重试: %s\n' "${TRAIN_DIR}" >&2
    exit 1
}
test -f "${BACKEND_DIR}/run_qwen3_1.7b_wordle_torchtitan_npu.sh"
test -f "${BACKEND_DIR}/asset_utils.py"
test -f "${TRAIN_DIR}/prepare_data.py"

python3 -m pip install --user 'uv==0.12.0'
export PATH="$(python3 -m site --user-base)/bin:${PATH}"
cd "${TRAIN_DIR}"
bash torchtitan_backend/setup_backend.sh

BACKEND_PYTHON="${BACKEND_DIR}/.venv/bin/python3"
MODELSCOPE="${BACKEND_DIR}/.venv/bin/modelscope"

assets_complete() {
    local asset_dir=$1
    test -f "${asset_dir}/models/Qwen3-1.7B-Wordle-SFT/config.json" &&
        compgen -G "${asset_dir}/models/Qwen3-1.7B-Wordle-SFT/*.safetensors" >/dev/null &&
        test -f "${asset_dir}/data/wordle_train.parquet" &&
        test -f "${asset_dir}/data/wordle_test.parquet"
}

if ASSET_CANDIDATE=$("${BACKEND_PYTHON}" "${BACKEND_DIR}/asset_utils.py" \
        --recipe-dir "${TRAIN_DIR}" 2>/dev/null) && assets_complete "${ASSET_CANDIDATE}"; then
    ASSET_RECIPE_DIR="${ASSET_CANDIDATE}"
    printf '复用已有训练资产: %s\n' "${ASSET_RECIPE_DIR}"
else
    ASSET_RECIPE_DIR="${TRAIN_DIR}"
    if [[ ! -f "${ASSET_RECIPE_DIR}/data/wordle_train.parquet" ||
          ! -f "${ASSET_RECIPE_DIR}/data/wordle_test.parquet" ]]; then
        "${BACKEND_PYTHON}" prepare_data.py \
            --seed 42 --num_train 2000 --num_test 20 --output_dir data
    fi

    MODEL_DIR="${ASSET_RECIPE_DIR}/models/Qwen3-1.7B-Wordle-SFT"
    if [[ ! -f "${MODEL_DIR}/config.json" ]] ||
       ! compgen -G "${MODEL_DIR}/*.safetensors" >/dev/null; then
        mkdir -p "${MODEL_DIR}"
        TORCH_DEVICE_BACKEND_AUTOLOAD=0 "${MODELSCOPE}" download \
            --model misumisumisu/Qwen3-1.7B-Wordle-SFT \
            --local_dir "${MODEL_DIR}"
    fi

    ASSET_RECIPE_DIR=$("${BACKEND_PYTHON}" "${BACKEND_DIR}/asset_utils.py" \
        --recipe-dir "${TRAIN_DIR}" --override "${TRAIN_DIR}")
fi

test -f "${ASSET_RECIPE_DIR}/models/Qwen3-1.7B-Wordle-SFT/config.json"
compgen -G "${ASSET_RECIPE_DIR}/models/Qwen3-1.7B-Wordle-SFT/*.safetensors" >/dev/null
test -f "${ASSET_RECIPE_DIR}/data/wordle_train.parquet"
test -f "${ASSET_RECIPE_DIR}/data/wordle_test.parquet"
printf '运行环境与训练资产准备完成: %s\n' "${ASSET_RECIPE_DIR}"


安装脚本在结束前执行后端验证，同一 Cell 随后完成资产查找、缺失项准备和最终完整性检查。安装与启动脚本均选择 ATB `--cxx_abi=1` 环境变体，使其与课程固定的 PyTorch wheel 所使用的 C++11 ABI 保持一致；这是二进制兼容条件，不是性能开关。后续 Cell 直接使用同一 launcher；包来源、NPU 状态、训练资产或其他关键条件不满足时，启动阶段会给出错误并终止。


## 课后练习

### 判断题

1. （判断题）`.venv` 保存 Python 环境，`runtime_sources` 保存需要固定和适配的上游源码。

2. （判断题）训练 launcher 应使用独立 `.venv` 中的 Python，而不是依赖当前 Notebook kernel 的包环境。

### 单选题

3. （单选题）环境准备 Cell 使用哪个工具管理独立 Python 环境？

   A. uv 0.12.0

   B. 系统 apt

   C. Docker Compose

   D. Conda base 环境

### 多选题

4. （多选题）环境准备脚本主要完成哪些工作？

   A. 获取固定源码

   B. 应用适配补丁

   C. 安装依赖

   D. 执行后端环境检查

5. （多选题）环境与资产准备 Cell 的幂等行为包括哪些？

   A. 复用已有训练代码目录

   B. 复用完整的模型和 parquet

   C. 仅在资产缺失时生成数据或下载模型

   D. 每次执行前删除已有模型


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/07_torchtitan_wordle_training/answer/07.02_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
